## 04 Agricultural Context: USDA NASS County Statistics
**Series:** Tribal Agriculture & Land Health in South Dakota  
**Author:** Lilly Jones, PhD  
**Primary Focus:** Oglala Lakota (Pine Ridge), Sicangu Lakota (Rosebud)  
**In Scope:** All South Dakota Tribal Nations  
**Data Sources:** USDA NASS QuickStats API, Census TIGER Counties

## Purpose
This notebook uses USDA NASS county-level agricultural statistics to establish
the regional agricultural context for South Dakota Tribal lands. NASS data is
the most complete publicly available record of livestock counts, hay production,
and land use in the counties overlapping Tribal boundaries.

## Critical Limitation: Read Before Interpreting Results
> **NASS data systematically undercounts Tribal agricultural activity.**
>
> NASS collects county-level data from farm operators. Federal trust land
> (reservation land held in trust by the BIA) is reported differently from
> fee simple land, and many Tribal agricultural operations, particularly
> subsistence operations and communally managed rangeland, are not captured
> in NASS census or survey frames.
>
> This means NASS data for counties like Oglala Lakota (Pine Ridge)
> and Todd (Rosebud Sioux) likely undercounts actual livestock and
> agricultural activity on Tribal land. The gap between what NASS reports
> and what Tribal programs know is exactly why Tribal-collected operational
> data (the pipeline track of this repo) matters.

## What This Notebook Does
Despite that limitation, NASS provides the only consistent long-term county-
level baseline. This notebook:
- Identifies the counties that overlap each SD Tribal Nation
- Pulls NASS livestock (cattle, horses), hay, and land use data for those counties
- Tracks trends over time
- Documents where NASS shows suppressed values `(D)`, which on reservations
  often means the activity exists but is withheld for confidentiality

## Commodities in Scope
| Commodity | Relevance |
|---|---|
| Cattle and calves | Primary livestock enterprise on most SD Tribal lands |
| Horses | Important to Lakota and other Nations; often undercounted |
| Hay | Primary forage crop; indicator of carrying capacity |
| Farms/land in farms | Baseline land use indicator |

Note: Bison are reported inconsistently in NASS: they appear under
'BISON' commodity but with heavy suppression in reservation counties.
Bison counts from Tribal programs are far more reliable than NASS.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
gpd.options.io_engine = "fiona"
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import zipfile, io, tempfile
from dotenv import load_dotenv
from shapely.validation import make_valid

load_dotenv(REPO_ROOT / ".env")

from src.data import constants
from src.data.constants import (
    SD_TRIBES_ALL, SD_TRIBES_PRIMARY,
    CENSUS_NAME_MAP, CENSUS_TO_COMMON,
    CRS_GEOGRAPHIC, CRS_PROJECTED,
)
from src.indigenous.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

print(f"Repo root : {REPO_ROOT}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["census_aiannh", "usda_nass"])

## Configure

In [ ]:
# API key and parameters
NASS_API_KEY = os.environ.get("NASS_API_KEY", None)
NASS_API_AVAILABLE = NASS_API_KEY is not None

NASS_BASE_URL = constants.USDA_NASS_API_BASE + "/api_GET/"

# Analysis period
START_YEAR = 2000
END_YEAR   = 2024

# Commodities targeted to SD Tribal agricultural context
NASS_QUERIES = [
    {
        "label":          "Cattle and Calves",
        "commodity_desc": "CATTLE",
        "statisticcat":   "INVENTORY",
        "short_desc":     "CATTLE, INCL CALVES: INVENTORY",
    },
    {
        "label":          "Horses",
        "commodity_desc": "HORSES",
        "statisticcat":   "INVENTORY",
        "short_desc":     None,  
    },
    {
        "label":          "Bison",
        "commodity_desc": "BISON",
        "statisticcat":   "INVENTORY",
        "short_desc":     None,
    },
    {
        "label":          "Hay",
        "commodity_desc": "HAY",
        "statisticcat":   "PRODUCTION",
        "short_desc":     None,
    },
    {
        "label":          "Land in Farms",
        "commodity_desc": "AG LAND",
        "statisticcat":   "AREA",
        "short_desc":     "AG LAND in ACRES",
    },
]

print("USDA NASS CONFIGURATION")
print(f"  API key     : {'SET' if NASS_API_AVAILABLE else 'NOT SET: get free key at https://quickstats.nass.usda.gov/api'}")
print(f"  Period      : {START_YEAR}–{END_YEAR}")
print(f"  Commodities : {', '.join(q['label'] for q in NASS_QUERIES)}")
print()
print("IMPORTANT: NASS data is county-level and undercounts Tribal")
print("agricultural activity on trust land. Results reflect the county,")
print("not exclusively the Tribal land within the county.")

## Load Tribal Boundaries and Overlapping Counties

In [ ]:
# Tribal boundaries 
GEOJSON_PATH = constants.OUTPUTS_DIR / "sd_tribal_land_base.geojson"
CACHE_PATH   = constants.CACHE_DIR   / "tl_2023_us_aiannh.geojson"

if GEOJSON_PATH.exists():
    tribal_lands = gpd.read_file(GEOJSON_PATH)
    print(f"Loaded tribal lands from notebook 01: {len(tribal_lands)}")
elif CACHE_PATH.exists():
    all_aiannh = gpd.read_file(CACHE_PATH)
    census_names = list(CENSUS_NAME_MAP.values())
    tribal_lands = all_aiannh[all_aiannh["NAME"].isin(census_names)].copy()
    tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False)
    tribal_lands["geometry"]    = tribal_lands.geometry.apply(make_valid)
    tribal_lands["common_name"] = tribal_lands["NAME"].map(CENSUS_TO_COMMON)
    tribal_lands["area_km2"]    = tribal_lands.to_crs(CRS_PROJECTED).geometry.area / 1e6
    tribal_lands["is_primary"]  = tribal_lands["common_name"].isin(SD_TRIBES_PRIMARY)
else:
    raise FileNotFoundError("Run notebook 01 first.")

In [ ]:
# Census TIGER county boundaries
COUNTY_CACHE = constants.CACHE_DIR / "tl_2023_us_county.geojson"

if COUNTY_CACHE.exists():
    all_counties = gpd.read_file(COUNTY_CACHE)
    print(f"Counties loaded from cache: {len(all_counties):,}")
else:
    print("Downloading Census TIGER county boundaries...")
    url = f"{constants.CENSUS_TIGER_BASE}/TIGER2023/COUNTY/tl_2023_us_county.zip"
    r   = requests.get(url, timeout=300)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        with tempfile.TemporaryDirectory() as tmp:
            z.extractall(tmp)
            shp = next(Path(tmp).glob("*.shp"))
            all_counties = gpd.read_file(shp).to_crs(CRS_GEOGRAPHIC)
    all_counties.to_file(COUNTY_CACHE, driver="GeoJSON")
    print(f"Downloaded and cached: {len(all_counties):,} counties")

# South Dakota counties only (FIPS state code 46)
sd_counties = all_counties[all_counties["STATEFP"] == "46"].copy().reset_index(drop=True)
sd_counties["county_name_upper"] = sd_counties["NAME"].str.upper()
print(f"South Dakota counties: {len(sd_counties)}")

In [ ]:
# Spatial join: Tribal lands/overlapping counties 
tribal_proj  = tribal_lands.to_crs(CRS_PROJECTED)
county_proj  = sd_counties.to_crs(CRS_PROJECTED)

tribe_county = gpd.sjoin(
    tribal_proj[["common_name", "is_primary", "geometry"]],
    county_proj[["NAME", "GEOID", "COUNTYFP", "county_name_upper", "geometry"]]
    .rename(columns={"NAME": "county_name"}),
    how="left",
    predicate="intersects",
).drop(columns=["index_right"], errors="ignore").reset_index(drop=True)

# Compute overlap area for context
overlap_areas = []
for _, tc in tribe_county.iterrows():
    tribe_geom  = tribal_proj[tribal_proj["common_name"] == tc["common_name"]].geometry.iloc[0]
    county_geom = county_proj[county_proj["GEOID"] == tc["GEOID"]].geometry.iloc[0]
    overlap_km2 = tribe_geom.intersection(county_geom).area / 1e6
    overlap_areas.append(overlap_km2)
tribe_county["overlap_km2"] = overlap_areas

# Unique (tribe, county) pairs for NASS queries
tribe_county_pairs = (
    tribe_county[["common_name", "is_primary", "county_name",
                  "COUNTYFP", "GEOID", "overlap_km2"]]
    .dropna(subset=["COUNTYFP"])
    .drop_duplicates(subset=["common_name", "GEOID"])
    .reset_index(drop=True)
)

print("TRIBAL NATION to COUNTY MAPPING")
for name, grp in tribe_county_pairs.groupby("common_name"):
    counties = grp["county_name"].tolist()
    flag = " ◄ PRIMARY" if name in SD_TRIBES_PRIMARY else ""
    print(f"  {name:<35}: {', '.join(counties)}{flag}")

# Unique NASS county list (COUNTYFP 3-digit strings)
nass_counties = tribe_county_pairs["COUNTYFP"].unique().tolist()
print(f"\nUnique counties to query from NASS: {len(nass_counties)}")

## Fetch USDA NASS Data
Queries the NASS QuickStats API for all counties overlapping SD Tribal lands.
Results are cached to `data/cache/`. Free API key required:
https://quickstats.nass.usda.gov/api

In [ ]:
# NASS API query function
def clean_nass_value(val):
    """
    Convert NASS Value string to float.
    (D) = withheld to avoid disclosing data for individual operations
    (Z) = less than half the unit shown
    Returns None for suppressed values.
    """
    if val in ["(D)", "(Z)", "(L)", "(NA)", None, ""]:
        return None
    try:
        return float(str(val).replace(",", ""))
    except ValueError:
        return None


def fetch_nass(
    api_key: str,
    commodity_desc: str,
    statisticcat_desc: str,
    state_alpha: str = "SD",
    start_year: int = START_YEAR,
    short_desc: str | None = None,
) -> pd.DataFrame:
    """
    Query USDA NASS QuickStats API for county-level data.
    Returns DataFrame with cleaned values and suppression flags.
    """
    params = {
        "key":                api_key,
        "state_alpha":        state_alpha,
        "agg_level_desc":     "COUNTY",
        "commodity_desc":     commodity_desc,
        "statisticcat_desc":  statisticcat_desc,
        "year__GE":           str(start_year),
        "format":             "JSON",
    }
    if short_desc:
        params["short_desc"] = short_desc

    r = requests.get(NASS_BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    if "data" not in data or not data["data"]:
        return pd.DataFrame()

    df = pd.DataFrame(data["data"])

    # Standardize key columns
    df["year"]              = pd.to_numeric(df.get("year"), errors="coerce")
    df["county_name_upper"] = df.get("county_name", pd.Series()).str.upper()
    df["county_fips"]       = df.get("state_fips_code", "") + df.get("county_code", "")
    df["value_clean"]       = df.get("Value", pd.Series()).apply(clean_nass_value)
    df["suppressed"]        = df.get("Value", pd.Series()).isin(["(D)", "(Z)", "(L)", "(NA)"])
    df["short_desc"]        = df.get("short_desc", "")
    df["unit_desc"]         = df.get("unit_desc", "")

    return df[[
        "year", "county_name_upper", "county_fips",
        "short_desc", "unit_desc", "value_clean", "suppressed"
    ]].dropna(subset=["year"]).reset_index(drop=True)

In [ ]:
# Download NASS data 
try:
    constants.CACHE_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

NASS_CACHE = constants.CACHE_DIR / "nass_sd_tribal_counties.csv"

if not NASS_API_AVAILABLE:
    print(
        "NASS API key not set. Get a free key at:\n"
        "  https://quickstats.nass.usda.gov/api\n"
        "Then add to .env: NASS_API_KEY=your_key\n\n"
        "If a cache file exists from a previous run, it will be used."
    )

if NASS_CACHE.exists():
    nass_raw = pd.read_csv(NASS_CACHE)
    nass_raw["year"] = pd.to_numeric(nass_raw["year"], errors="coerce")
    print(f"Loaded NASS data from cache: {len(nass_raw):,} records")

elif NASS_API_AVAILABLE:
    all_parts = []
    for q in NASS_QUERIES:
        print(f"  Fetching {q['label']}...")
        try:
            df = fetch_nass(
                api_key=NASS_API_KEY,
                commodity_desc=q["commodity_desc"],
                statisticcat_desc=q["statisticcat"],
                short_desc=q.get("short_desc"),
            )
            if not df.empty:
                df["label"] = q["label"]
                # Filter to Tribal-overlapping counties only
                df = df[
                    df["county_name_upper"].isin(
                        tribe_county_pairs["county_name"].str.upper()
                    )
                ]
                all_parts.append(df)
                print(f"    → {len(df):,} records")
            else:
                print(f"    → No data returned")
        except Exception as e:
            print(f"    → FAILED: {e}")

    if all_parts:
        nass_raw = pd.concat(all_parts, ignore_index=True)
        nass_raw.to_csv(NASS_CACHE, index=False)
        print(f"\nCached {len(nass_raw):,} total records")
    else:
        raise RuntimeError("No NASS data returned. Check API key and network access.")

else:
    raise RuntimeError(
        "No NASS data available. Set NASS_API_KEY in .env to download."
    )

## Join to Tribal Lands

In [ ]:
# Link NASS counties to Tribal Nations 
# One county can overlap multiple Tribes; one Tribe can span multiple counties.
# Attribution: assign NASS values to each Tribe overlapping that county.
# This is a known limitation: the data is not Tribe-exclusive.

county_tribe_lookup = (
    tribe_county_pairs[["common_name", "is_primary", "county_name", "overlap_km2"]]
    .assign(county_name_upper=lambda df: df["county_name"].str.upper())
)

nass_tribal = nass_raw.merge(
    county_tribe_lookup,
    on="county_name_upper",
    how="inner",
)

# Deduplicate: if a county row maps to multiple Tribes, keep all (intentional)
print(f"NASS records linked to Tribal Nations: {len(nass_tribal):,}")
print(f"\nSuppression rate by commodity (% of county-years withheld as (D)):")
suppression = (
    nass_tribal.groupby("label")["suppressed"]
    .agg(["sum", "count"])
    .assign(pct_suppressed=lambda df: (df["sum"] / df["count"] * 100).round(1))
    .reset_index()
    .rename(columns={"sum": "suppressed_count", "count": "total_records"})
)
print(suppression[["label", "suppressed_count", "total_records", "pct_suppressed"]].to_string(index=False))
print()
print("High suppression rates (especially for Bison) indicate that activity")
print("exists but is withheld to protect individual operator confidentiality.")
print("Suppression is NOT the same as absence.")

## Analysis

In [ ]:
# Mean livestock counts by Tribal Nation
# For Tribes spanning multiple counties, take the mean across counties
# weighted by overlap area. Where all values are suppressed, report as such.

def weighted_mean(grp):
    """Weighted mean by overlap_km2; returns None if all values suppressed."""
    valid = grp.dropna(subset=["value_clean"])
    if valid.empty:
        return None
    weights = valid["overlap_km2"].fillna(1)
    return np.average(valid["value_clean"], weights=weights)


tribal_ag = (
    nass_tribal.groupby(["common_name", "is_primary", "year", "label"])
    .apply(weighted_mean)
    .reset_index()
    .rename(columns={0: "value"})
)

tribal_ag_summary = (
    tribal_ag.groupby(["common_name", "is_primary", "label"])
    .agg(
        mean_value=("value", "mean"),
        min_value=("value",  "min"),
        max_value=("value",  "max"),
        pct_missing=("value", lambda x: x.isna().mean() * 100),
    )
    .round(0)
    .reset_index()
)

print("MEAN AGRICULTURAL STATISTICS BY TRIBAL NATION")
print(f"({START_YEAR}–{END_YEAR}, county-weighted by overlap area)")
print("=" * 65)
for label, grp in tribal_ag_summary.groupby("label"):
    print(f"\n{label}:")
    print(
        grp[["common_name", "mean_value", "pct_missing"]]
        .sort_values("mean_value", ascending=False)
        .to_string(index=False)
    )
    print("  * pct_missing = % of year-observations with suppressed (D) values")

In [ ]:
# Statewide SD trend, all Tribal counties combined
sd_trend = (
    nass_tribal[nass_tribal["value_clean"].notna()]
    .groupby(["year", "label"])["value_clean"]
    .sum()
    .reset_index()
    .rename(columns={"value_clean": "total"})
)

print("TREND SUMMARY: Tribal-County Aggregate")
print("=" * 50)
for label, grp in sd_trend.groupby("label"):
    grp = grp.sort_values("year")
    if len(grp) < 2:
        continue
    first_yr = grp.iloc[0]
    last_yr  = grp.iloc[-1]
    pct_chg  = (last_yr["total"] - first_yr["total"]) / first_yr["total"] * 100
    direction = "↑" if pct_chg > 0 else "↓"
    print(f"  {label:<20}: {first_yr['year']:.0f}={first_yr['total']:>10,.0f} → "
          f"{last_yr['year']:.0f}={last_yr['total']:>10,.0f}  "
          f"{direction} {abs(pct_chg):.1f}%")

## Visualizations

In [ ]:
# Time series: cattle and hay for primary Tribes
PRIMARY_COLORS = {
    "Oglala Lakota": "#C0392B",
    "Rosebud Sioux": "#1A5276",
}

for label in ["Cattle & Calves", "Hay"]:
    fig, ax = plt.subplots(figsize=(12, 5))

    for name, color in PRIMARY_COLORS.items():
        grp = tribal_ag[
            (tribal_ag["common_name"] == name) &
            (tribal_ag["label"] == label)
        ].sort_values("year").dropna(subset=["value"])
        if grp.empty:
            continue
        ax.plot(grp["year"], grp["value"],
                color=color, linewidth=2, marker="o",
                markersize=4, label=name)

    unit = "head" if label != "Hay" else "tons"
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x:,.0f}"
    ))
    ax.set_xlabel("Year", fontsize=10)
    ax.set_ylabel(f"{label} ({unit})", fontsize=10)
    ax.set_title(
        f"{label} Pine Ridge and Rosebud Counties\n"
        "USDA NASS county-level data (county overlapping Tribal land)",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9)
    ax.text(
        0.01, 0.03,
        "Note: County-level data includes non-Tribal land. "
        "Tribal agricultural activity may be undercounted.",
        transform=ax.transAxes, fontsize=7, color="gray",
        ha="left", va="bottom",
    )
    sns.despine(ax=ax)
    plt.tight_layout()

    try:
        fig_dir = constants.OUTPUTS_DIR / "figures"
        fig_dir.mkdir(parents=True, exist_ok=True)
        safe_label = label.lower().replace(" ", "_").replace("&", "and")
        fig.savefig(fig_dir / f"04_nass_{safe_label}_time_series.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

In [ ]:
# Bar chart: cattle inventory comparison for all SD Tribes 
cattle_summary = tribal_ag_summary[
    tribal_ag_summary["label"] == "Cattle & Calves"
].sort_values("mean_value", ascending=True)

if not cattle_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [
        PRIMARY_COLORS.get(n, "#566573")
        for n in cattle_summary["common_name"]
    ]
    bars = ax.barh(
        cattle_summary["common_name"],
        cattle_summary["mean_value"],
        color=colors, alpha=0.85,
    )
    # Suppression flag
    for bar, (_, row) in zip(bars, cattle_summary.iterrows()):
        if row["pct_missing"] > 50:
            ax.text(
                bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
                f"⚠ {row['pct_missing']:.0f}% suppressed",
                va="center", fontsize=7, color="#E67E22",
            )

    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x:,.0f}"
    ))
    ax.set_xlabel("Mean cattle inventory (head)", fontsize=10)
    ax.set_title(
        f"Cattle Inventory by Tribal Nation\n"
        f"NASS county-level mean {START_YEAR}–{END_YEAR} (county overlapping Tribal land)",
        fontsize=11, fontweight="bold",
    )
    ax.legend(
        handles=[
            mpatches.Patch(color="#C0392B", label="Primary focus"),
            mpatches.Patch(color="#566573", label="South Dakota Tribes"),
        ],
        fontsize=8,
    )
    sns.despine(ax=ax)
    plt.tight_layout()
    try:
        fig.savefig(fig_dir / "04_nass_cattle_comparison.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

In [ ]:
# Suppression map: where is NASS data being withheld? 
# High suppression rates in reservation counties suggest agricultural activity
# that exists but cannot be measured through NASS.

# County-level suppression rate for cattle
cattle_suppression = (
    nass_tribal[nass_tribal["label"] == "Cattle & Calves"]
    .groupby("county_name_upper")["suppressed"]
    .agg(["sum", "count"])
    .assign(pct_suppressed=lambda df: df["sum"] / df["count"] * 100)
    .reset_index()
)

counties_plot = sd_counties.merge(
    cattle_suppression,
    on="county_name_upper",
    how="left",
)
counties_plot["pct_suppressed"] = counties_plot["pct_suppressed"].fillna(0)

fig, ax = plt.subplots(figsize=(12, 7))

counties_plot.to_crs(3857).plot(
    column="pct_suppressed", cmap="YlOrRd",
    ax=ax, edgecolor="white", linewidth=0.5,
    legend=True,
    legend_kwds={"label": "% of cattle records suppressed (D)", "shrink": 0.6},
)

# Tribal boundaries overlay
tribal_lands.to_crs(3857).boundary.plot(
    ax=ax, color="#1A5276", linewidth=2, label="Tribal land boundary"
)

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.3)
except Exception:
    pass

ax.set_axis_off()
ax.set_title(
    "NASS Cattle Data Suppression Rate by County\n"
    "High suppression in reservation counties ≠ absence of cattle",
    fontsize=11, fontweight="bold",
)
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
try:
    fig.savefig(fig_dir / "04_nass_suppression_map.png",
                dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

## Exports

In [ ]:
try:
    constants.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

tribal_ag_summary.to_csv(
    constants.OUTPUTS_DIR / "nass_tribal_ag_summary.csv", index=False
)
print("Exported to outputs/nass_tribal_ag_summary.csv")

tribe_county_pairs.to_csv(
    constants.OUTPUTS_DIR / "tribal_county_overlap.csv", index=False
)
print("Exported to outputs/tribal_county_overlap.csv")

suppression.to_csv(
    constants.OUTPUTS_DIR / "nass_suppression_by_commodity.csv", index=False
)
print("Exported to outputs/nass_suppression_by_commodity.csv")

## Summary and Findings

*(Fill in after running with real data.)*

**What the data shows:**
- Cattle inventory in the counties overlapping Pine Ridge and Rosebud, does it trend
  up, down, or stable since 2000?
- How does the suppression rate for Bison compare to Cattle? Heavy Bison
  suppression in reservation counties confirms that Tribal bison programs
  exist but cannot be quantified through NASS.
- What is the hay production trend in Pine Ridge and Rosebud counties?
  Declining hay production in drought years should correlate with the PDSI
  and NDVI records from notebooks 02 and 03.

**The suppression finding:**
The suppression map is potentially the most important output in this notebook.
If reservation counties show systematically higher suppression rates than
surrounding counties, that is evidence of a federal data infrastructure gap,
not of less agricultural activity. 

**Why Tribal-collected data matters:**
NASS county data answers: "What is the agricultural context for the region?"
Tribal operational data answers: "What is actually happening on our land, in
our pastures, with our herds?" The gap between those two questions is the
motivation for the pipeline track in this repository.

**Connection to the rest of the series:**
- Cross-reference cattle inventory trend with NDVI from notebook 03. 
  Do livestock numbers decline in the same years that NDVI drops?
- Notebook 05 (water) will add well-level groundwater context that affects
  carrying capacity for livestock
- Notebook 06 (system stress) combines drought + vegetation + these
  agricultural indicators into a single compound stress index

In [ ]:
# Print citations
print(generate_citations(["census_aiannh", "usda_nass"]))